# Fix System G (BIO) language-ablation matrix

Clean run. Cell 1 sets up Drive symlinks **correctly** (the old notebook's `ln -sfn` nested instead of replacing, because the repo commits real `models/`+`results/` dirs, so all output went to ephemeral `/content` and died on reset).

**Do not run any training cell until Cell 1's gate prints `islink: True` for both.**

In [ ]:
# Cell 1 - Mount, clone, symlink (FIXED), hard gate
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys, os, shutil

REPO = '/content/Research_And_Training'
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone',
    'https://github.com/JustLetMeBeHello/Idiomator_Research.git', REPO], check=True)

os.makedirs('/content/drive/MyDrive/Idiomator_Research/models',  exist_ok=True)
os.makedirs('/content/drive/MyDrive/Idiomator_Research/results', exist_ok=True)

# Remove the real committed dirs FIRST, then symlink. ln/-sfn will NOT replace a
# populated real dir - it nests inside it. shutil.rmtree guarantees a clean target.
for d in ['models', 'results']:
    p = os.path.join(REPO, d)
    if os.path.islink(p):
        os.unlink(p)
    elif os.path.isdir(p):
        shutil.rmtree(p)
    os.symlink(f'/content/drive/MyDrive/Idiomator_Research/{d}', p)

# HARD GATE - both must be True or STOP (do not train)
ok = True
for d in ['models', 'results']:
    p = os.path.join(REPO, d)
    is_link = os.path.islink(p)
    ok &= is_link
    print(d, 'islink:', is_link, '->', os.readlink(p) if is_link else '(REAL DIR - BAD)')
assert ok, 'Symlinks NOT set up - STOP, do not run training cells.'
print('\nGate passed. Safe to proceed.')

In [ ]:
# Cell 2 - Deps
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
    '/content/Research_And_Training/Requirements.txt'], check=True)
!nvidia-smi

In [ ]:
# Cell 3 - Run the System-G fix (delete stale sentinels -> retrain BIO 15 combos
#          -> G-only eval -> patch -> regenerate CSVs). ~3-4h. Tee log to Drive.
import subprocess, sys, os
LOG = '/content/drive/MyDrive/Idiomator_Research/fix_system_g.log'
with open(LOG, 'w') as lf:
    proc = subprocess.Popen(
        [sys.executable, '-u', 'Google_Colab/fix_system_g_combos.py'],
        cwd='/content/Research_And_Training',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in proc.stdout:
        print(line, end='', flush=True)
        lf.write(line); lf.flush()
    proc.wait()
print('\nexit code:', proc.returncode)

In [ ]:
# Cell 4 - Verify g_only outputs actually persisted on DRIVE (not ephemeral)
from pathlib import Path
base = Path('/content/drive/MyDrive/Idiomator_Research/results/language_ablation_matrix')
combos = ['en','es','hi','te','en_es','en_hi','en_te','es_hi','es_te','hi_te',
          'en_es_hi','en_es_te','en_hi_te','es_hi_te','en_es_hi_te']
n = 0
for c in combos:
    f = base / c / 'g_only' / 'pipeline_eval_results.json'
    print('OK ' if f.exists() else 'MISSING', c)
    n += f.exists()
print(f'\n{n}/15 g_only results on Drive')
assert n == 15, 'Not all 15 persisted - check Cell 1 gate and Cell 3 log.'

In [ ]:
# Cell 5 - Aggregate G matrix + compare vs QA systems
import subprocess, sys, os
proc = subprocess.Popen(
    [sys.executable, '-u', 'Ablations/summarize_g_only.py',
     '--g_dir',  '/content/drive/MyDrive/Idiomator_Research/results/language_ablation_matrix',
     '--qa_dir', '/content/drive/MyDrive/IdiomBERT_Ablations/results'],
    cwd='/content/Research_And_Training',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()